# 27. Khai báo trước giả thuyết và cổng kiểm tra

Báo cáo tuần 3 chỉ ra một điểm yếu về suy luận: họ 38 phép kiểm định được gom sau khi đã thấy kết quả, nên hiệu chỉnh Holm chỉ có giá trị thận trọng chứ không phải xác nhận. Sổ tay này khóa họ kiểm định của tuần 4 trước khi bất kỳ mô hình mới nào được huấn luyện.

Năm việc:

- Kiểm cổng giao thức: 136.752 điểm, mốc Persistence 0,578754.
- Kiểm thứ tự chạy của tuần 3: gói hợp nhất notebook 22 đến notebook 26 phải mới hơn ba notebook bổ sung 19, 20 và 21.
- Kiểm mảng đặc trưng tĩnh S do notebook 03 ghi: đủ chiều, không có giá trị không hữu hạn.
- Kiểm các lần chạy nền của tuần 3 sẽ dùng làm đối chứng.
- Ghi tệp khai báo trước gồm sáu phép kiểm định, quy tắc kết luận H2 và tiêu chí chọn ngưỡng Huber, kèm mã băm SHA-256. Các sổ tay sau đọc tệp này; notebook 31 kiểm lại mã băm trước khi hiệu chỉnh Holm.

Không huấn luyện mô hình nào. Chạy dưới hai phút, không cần bộ xử lý đồ họa.

> Thứ tự chạy tuần 4: notebook 27, notebook 28, notebook 29, notebook 30, notebook 31. Kết quả ghi vào thư mục con `Output/processed/w4_34_khai_bao_truoc_va_cong_tuan_4`.

> **Vị trí trong chuỗi tái lập:** notebook 27 trên tổng số 31. Thứ tự chạy và sơ đồ phụ thuộc nằm trong `README.md`.
>
> **Cần chạy trước:** 03-Tien_xu_ly_va_thuc_nghiem_chinh, 22-Hop_nhat_chi_so.
>
> **Thư mục kết quả:** `Output/processed/w4_34_khai_bao_truoc_va_cong_tuan_4/`.
>
> **Mã định danh nội bộ:** `W4_34`. Mã này còn xuất hiện trong tên khóa JSON, tên hình và nhật ký chạy; nó được giữ nguyên để không phá phép đối chiếu chéo và mã băm của bản khai báo trước.

## 1. Tham số và đường dẫn

In [1]:
#@title Tham số chạy { display-mode: "form" }
PROJECT_DIR = "/content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4"  #@param {type:"string"}
READ_TAG = "chot"  #@param {type:"string"}
RUN_TAG = "w4_34_khai_bao_truoc_va_cong_tuan_4"  #@param {type:"string"}
CHO_PHEP_GHI_DE_KHAI_BAO = False  #@param {type:"boolean"}
N_BOOT = 5000  #@param {type:"integer"}
HAT_GIONG_LAY_MAU = 0  #@param {type:"integer"}
MUC_ALPHA = 0.05  #@param {type:"number"}
MSE_MOC_KY_VONG = 0.5787542076552471  #@param {type:"number"}
TONG_DIEM_KY_VONG = 136752  #@param {type:"integer"}
CHAY_LAI_TU_DAU = False  #@param {type:"boolean"}
SEQ_LEN = 36     #@param {type:"integer"}
PRED_LEN = 12    #@param {type:"integer"}
N_SEEDS = 5      #@param {type:"integer"}
EPOCHS = 60      #@param {type:"integer"}
PATIENCE = 10    #@param {type:"integer"}
BATCH = 128      #@param {type:"integer"}
LR = 1e-3        #@param {type:"number"}

# Mọi sổ tay tuần 4 dùng năm hạt giống, nên giá trị dưới đây ghi đè tham số chung.
N_SEEDS = 5
import os, sys, json, time, random, datetime, platform, importlib, subprocess, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
else:
    PROJECT_DIR = os.environ.get("P12_PROJECT", "./AIO2026_Conquer_M4_DLinear_NLinear_Slot4")

PROJECT = Path(PROJECT_DIR)
OUT = PROJECT / "Output"
# Đọc dữ liệu đã tiền xử lý của lần chạy chốt, ghi kết quả sang một thư mục con riêng.
PROC = OUT / "processed" / READ_TAG
RUN_DIR = OUT / "processed" / RUN_TAG
RUNS = RUN_DIR / "runs"
FIGS = RUN_DIR / "figures"
LOGS = RUN_DIR / "logs"
for d in (RUN_DIR, RUNS, FIGS, LOGS):
    d.mkdir(parents=True, exist_ok=True)

# Tệp khai báo trước họ kiểm định của tuần 4, do W4_34 ghi.
KHAI_BAO_DIR = OUT / "processed" / "w4_34_khai_bao_truoc_va_cong_tuan_4"
TEP_KHAI_BAO = KHAI_BAO_DIR / "W4_khai_bao_ho_kiem_dinh.json"

assert (PROC / "meta.json").exists(), (
    "Không thấy dữ liệu đã tiền xử lý trong " + str(PROC) +
    ". Hãy chạy notebook 03-Tien_xu_ly_va_thuc_nghiem_chinh trước.")
meta = json.loads((PROC / "meta.json").read_text())
channels = meta["channels"]
TEN_TINH = list(meta.get("static", []))
EPS = 1e-8

print("Thư mục đọc :", PROC)
print("Thư mục ghi :", RUN_DIR)
print("Số kênh     :", len(channels), "| đặc trưng tĩnh:", TEN_TINH)

Mounted at /content/drive
Thư mục đọc : /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/processed/chot
Thư mục ghi : /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/processed/w4_34_khai_bao_truoc_va_cong_tuan_4
Số kênh     : 37 | đặc trưng tĩnh: ['Age', 'Gender', 'Height', 'ICUType_1', 'ICUType_2', 'ICUType_3', 'ICUType_4']


In [2]:
REQUIRED = {"numpy": "1.24.0", "torch": "2.0.0", "matplotlib": "3.7.0", "scipy": "1.10.0"}


def ver_tuple(v):
    out = []
    for part in str(v).split(".")[:3]:
        digits = "".join(ch for ch in part if ch.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out + [0] * (3 - len(out)))


missing = []
for name, floor in REQUIRED.items():
    try:
        mod = importlib.import_module(name)
        if ver_tuple(mod.__version__) < ver_tuple(floor):
            missing.append(name + ">=" + floor)
    except ImportError:
        missing.append(name + ">=" + floor)

if missing:
    print("Cài bổ sung:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import scipy

ENV = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "run_tag": RUN_TAG,
    "read_tag": READ_TAG,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "torch": torch.__version__,
    "matplotlib": matplotlib.__version__,
    "scipy": scipy.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
(LOGS / ("env_" + stamp + ".json")).write_text(json.dumps(ENV, ensure_ascii=False, indent=2))
for k, v in ENV.items():
    print(k.ljust(12), v)

timestamp    2026-10-14T18:58:52
run_tag      w4_34_khai_bao_truoc_va_cong_tuan_4
read_tag     chot
python       3.13.15
numpy        2.1.3
torch        2.11.0+cu130
matplotlib   3.10.0
scipy        1.16.3
gpu          Tesla T4


In [3]:
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]

matplotlib.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "legend.frameon": False,
    "lines.linewidth": 1.4,
    "axes.prop_cycle": matplotlib.cycler(color=PALETTE),
    "figure.constrained_layout.use": True,
})


def luu_hinh(fig, ten):
    duong = FIGS / ten
    fig.savefig(duong)
    fig.savefig(duong.with_suffix(".pdf"))
    print("Đã lưu", duong, "và bản vector pdf")
    return duong


print("Đã đặt quy ước trình bày hình")

Đã đặt quy ước trình bày hình


## 2. Mô hình, dữ liệu và hàm dùng chung

In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def moving_avg(x, kernel_size=25):
    front = x[:, :1, :].repeat(1, (kernel_size - 1) // 2, 1)
    end = x[:, -1:, :].repeat(1, kernel_size - (kernel_size - 1) // 2 - 1, 1)
    xp = torch.cat([front, x, end], dim=1).permute(0, 2, 1)
    return F.avg_pool1d(xp, kernel_size=kernel_size, stride=1).permute(0, 2, 1)


def last_observed(x, mask):
    # Trả về giá trị quan sát thật gần nhất và số giờ đã trôi qua kể từ phép đo đó.
    B, Lx, C = x.shape
    pos = torch.arange(Lx, device=x.device).view(1, Lx, 1).expand(B, Lx, C)
    idx = torch.where(mask > 0, pos, torch.full_like(pos, -1))
    t_star = idx.max(dim=1).values
    has = t_star >= 0
    val = torch.gather(x, 1, t_star.clamp(min=0).unsqueeze(1)).squeeze(1)
    val = torch.where(has, val, torch.zeros_like(val))
    dt = torch.where(has, (Lx - 1 - t_star).float(), torch.full_like(val, float(Lx)))
    return val, dt


# Tuần 4 chỉ dùng neo B trên giao thức chính. Định nghĩa A được giữ để kiểm tra chéo với tuần 3.
NEO_HOP_LE = ("A", "B")


class LinearFamily(nn.Module):
    # Lớp bao trùm DLinear và NLinear, chép nguyên công thức của tuần 3 cho hai neo A và B.
    def __init__(self, seq_len, pred_len, n_channels, kind="nlinear", anchor="B",
                 use_mask_channel=True, kernel_size=25):
        super().__init__()
        assert kind in ("dlinear", "nlinear")
        assert anchor in NEO_HOP_LE, "Neo không hợp lệ: " + str(anchor)
        self.L, self.H, self.C = seq_len, pred_len, n_channels
        self.kind, self.anchor = kind, anchor
        self.use_mask_channel = use_mask_channel
        self.kernel_size = kernel_size
        in_dim = seq_len * (2 if use_mask_channel else 1)
        if kind == "dlinear":
            self.lin_seasonal = nn.Linear(in_dim, pred_len)
            self.lin_trend = nn.Linear(in_dim, pred_len)
        else:
            self.lin = nn.Linear(in_dim, pred_len)

    def compute_anchor(self, x, mask):
        if self.anchor == "A":
            return x[:, -1, :]
        val, _ = last_observed(x, mask)
        return val

    def _flatten(self, x, mask):
        B, Lx, C = x.shape
        xt = x.permute(0, 2, 1).reshape(B * C, Lx)
        if not self.use_mask_channel:
            return xt
        mt = mask.permute(0, 2, 1).reshape(B * C, Lx).float()
        return torch.cat([xt, mt], dim=-1)

    def phan_du(self, x, mask):
        # Phần mô hình học, tính trên chuỗi đã trừ neo; trả về kích thước lô, H, C.
        B, Lx, C = x.shape
        a = self.compute_anchor(x, mask)
        xr = x - a.unsqueeze(1)
        if self.kind == "nlinear":
            z = self.lin(self._flatten(xr, mask))
        else:
            trend = moving_avg(xr, self.kernel_size)
            z = self.lin_seasonal(self._flatten(xr - trend, mask)) + \
                self.lin_trend(self._flatten(trend, mask))
        return z.view(B, C, self.H).permute(0, 2, 1), a

    def forward(self, x, mask):
        z, a = self.phan_du(x, mask)
        return z + a.unsqueeze(1)


class LinearFamilyTinh(LinearFamily):
    # Nhánh tĩnh đúng như giả thuyết H2 của tuần 1: hai hệ số vô hướng alpha và beta là hàm
    # Affine của vectơ đặc trưng tĩnh s, nên số tham số thêm đúng bằng 2(d_s + 1).
    # Điều chỉnh đặt lên phần dư sau khi trừ neo, nên neo B giữ nguyên mức nền của từng hồ sơ:
    # y_hat = a + (1 + alpha(s)) * z + beta(s).
    # Hai lớp tĩnh khởi tạo bằng không, lớp tuyến tính chính được tạo trước với cùng hạt giống,
    # Nên tại bước đầu mô hình trùng khít mô hình nền cùng hạt giống.
    def __init__(self, seq_len, pred_len, n_channels, d_static, kind="nlinear", anchor="B", **kw):
        super().__init__(seq_len, pred_len, n_channels, kind=kind, anchor=anchor, **kw)
        self.d_static = int(d_static)
        self.static_alpha = nn.Linear(self.d_static, 1)
        self.static_beta = nn.Linear(self.d_static, 1)
        for lop in (self.static_alpha, self.static_beta):
            nn.init.zeros_(lop.weight)
            nn.init.zeros_(lop.bias)

    def forward(self, x, mask, s):
        z, a = self.phan_du(x, mask)
        B = x.shape[0]
        alpha = self.static_alpha(s).view(B, 1, 1)
        beta = self.static_beta(s).view(B, 1, 1)
        return a.unsqueeze(1) + (1.0 + alpha) * z + beta


class Persistence(nn.Module):
    # Đường cơ sở không tham số: giữ nguyên phép đo thật gần nhất cho mọi bước tương lai.
    def __init__(self, pred_len):
        super().__init__()
        self.H = pred_len

    def forward(self, x, mask):
        val, _ = last_observed(x, mask)
        return val.unsqueeze(1).repeat(1, self.H, 1)


def du_doan(model, x, m, s):
    # Một điểm gọi duy nhất cho mọi loại mô hình, để vòng huấn luyện không rẽ nhánh.
    if isinstance(model, LinearFamilyTinh):
        return model(x, m, s)
    return model(x, m)


def dat_hat_giong(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def tong_theo_ho_so(y_hat, y, m):
    m = m.astype(np.float64)
    se = (m * (y_hat - y) ** 2).reshape(len(m), -1).sum(axis=1)
    ae = (m * np.abs(y_hat - y)).reshape(len(m), -1).sum(axis=1)
    dem = m.reshape(len(m), -1).sum(axis=1)
    return se, ae, dem


def khoang_tin_cay(se, ae, dem, n_boot=2000, seed=0, alpha=0.05):
    # Lấy mẫu lại ở cấp bệnh nhân, vì các điểm trong cùng một hồ sơ không độc lập với nhau.
    rng = np.random.default_rng(seed)
    N = len(dem)
    mses, maes = np.empty(n_boot), np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, N, N)
        mau = dem[idx].sum() + EPS
        mses[b] = se[idx].sum() / mau
        maes[b] = ae[idx].sum() / mau
    lo, hi = 100 * alpha / 2, 100 * (1 - alpha / 2)
    return {
        "mse": float(se.sum() / (dem.sum() + EPS)),
        "mse_ci": [float(np.percentile(mses, lo)), float(np.percentile(mses, hi))],
        "mae": float(ae.sum() / (dem.sum() + EPS)),
        "mae_ci": [float(np.percentile(maes, lo)), float(np.percentile(maes, hi))],
        "mae_theo_ho_so": float(np.mean(ae / np.maximum(dem, 1.0))),
    }


print("Thiết bị tính toán:", DEVICE)

Thiết bị tính toán: cuda


In [5]:
def nap_lat(ten, L, H):
    d = np.load(PROC / (ten + ".npz"))
    assert "S" in d.files, "Tệp " + ten + ".npz không có mảng đặc trưng tĩnh S"
    Xd, Md, Sd = d["X"], d["M"], d["S"]
    assert Xd.shape[0] == Sd.shape[0], "Số hồ sơ của X và S không khớp ở " + ten
    assert Xd.shape[1] >= L + H, "Lưới thời gian ngắn hơn L cộng H"
    x, mxx = Xd[:, :L], Md[:, :L]
    y, myy = Xd[:, L:L + H], Md[:, L:L + H]
    # Giữ đúng quy tắc lọc của tuần 2 và tuần 3: bỏ hồ sơ không có mục tiêu quan sát thật.
    giu = myy.reshape(len(myy), -1).sum(axis=1) > 0
    return (torch.from_numpy(x[giu]).float(), torch.from_numpy(mxx[giu]).float(),
            torch.from_numpy(y[giu]).float(), torch.from_numpy(myy[giu]).float(),
            torch.from_numpy(np.asarray(Sd[giu], dtype=np.float32)),
            int(giu.sum()), int((~giu).sum()))


SPLITS = {}
for ten in ("train", "val", "test"):
    x, mxx, y, myy, s, giu, bo = nap_lat(ten, SEQ_LEN, PRED_LEN)
    SPLITS[ten] = TensorDataset(x, mxx, y, myy, s)
    print(ten.ljust(6), "giữ", giu, "hồ sơ, bỏ", bo, "hồ sơ không có mục tiêu quan sát thật")

N_CH = SPLITS["train"].tensors[0].shape[-1]
D_STATIC = SPLITS["train"].tensors[4].shape[-1]
assert D_STATIC == len(TEN_TINH), "Số chiều S khác danh sách đặc trưng tĩnh trong meta.json"
print("Số kênh:", N_CH, "| số chiều đặc trưng tĩnh:", D_STATIC)

train  giữ 5559 hồ sơ, bỏ 41 hồ sơ không có mục tiêu quan sát thật
val    giữ 794 hồ sơ, bỏ 6 hồ sơ không có mục tiêu quan sát thật
test   giữ 1585 hồ sơ, bỏ 15 hồ sơ không có mục tiêu quan sát thật
Số kênh: 37 | số chiều đặc trưng tĩnh: 7


In [6]:
def tinh_mat_mat(pred, y, m, loai, delta=1.0):
    # Mọi hàm mất mát chỉ tính trên các ô có phép đo thật, mẫu số là tổng mặt nạ.
    if loai == "mse":
        thanh_phan = (pred - y) ** 2
    elif loai == "mae":
        thanh_phan = (pred - y).abs()
    elif loai == "huber":
        thanh_phan = F.huber_loss(pred, y, reduction="none", delta=delta)
    else:
        raise ValueError("Không biết loại mất mát " + str(loai))
    return (m * thanh_phan).sum() / (m.sum() + EPS)


def ten_delta(delta):
    return "d" + ("%.2f" % float(delta)).rstrip("0").rstrip(".").replace(".", "p")


def chay_mot_cau_hinh(kind, anchor="B", seed=0, nhan=None, loai_mat_mat="mse",
                      huber_delta=1.0, dung_tinh=False, xao_tron_tinh=False,
                      lat=None, thu_muc=None, ghi_file=True):
    """Huấn luyện một cấu hình, đánh giá trên tập kiểm tra, ghi kết quả ra thư mục runs."""
    lat = SPLITS if lat is None else lat
    thu_muc = RUNS if thu_muc is None else thu_muc
    dat_hat_giong(seed)

    if kind == "persistence":
        model = Persistence(PRED_LEN).to(DEVICE)
    elif dung_tinh:
        model = LinearFamilyTinh(SEQ_LEN, PRED_LEN, N_CH, D_STATIC, kind=kind, anchor=anchor).to(DEVICE)
    else:
        model = LinearFamily(SEQ_LEN, PRED_LEN, N_CH, kind=kind, anchor=anchor).to(DEVICE)
    so_tham_so = sum(p.numel() for p in model.parameters() if p.requires_grad)

    loaders = {k: DataLoader(v, batch_size=BATCH, shuffle=(k == "train"))
               for k, v in lat.items()}

    t0 = time.time()
    tot_nhat, trang_thai, xau, lich_su = float("inf"), None, 0, []
    val_mse_tot, val_mae_tot = None, None
    if kind != "persistence":
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        for ep in range(EPOCHS):
            model.train()
            for x, mxx, y, myy, s in loaders["train"]:
                x, mxx, y, myy, s = [t.to(DEVICE) for t in (x, mxx, y, myy, s)]
                opt.zero_grad()
                mat_mat = tinh_mat_mat(du_doan(model, x, mxx, s), y, myy, loai_mat_mat, huber_delta)
                mat_mat.backward()
                opt.step()
            model.eval()
            with torch.no_grad():
                tu = tu_mse = tu_mae = mau = 0.0
                for x, mxx, y, myy, s in loaders["val"]:
                    x, mxx, y, myy, s = [t.to(DEVICE) for t in (x, mxx, y, myy, s)]
                    p = du_doan(model, x, mxx, s)
                    tu += float(tinh_mat_mat(p, y, myy, loai_mat_mat, huber_delta)) * float(myy.sum() + EPS)
                    tu_mse += float((myy * (p - y) ** 2).sum())
                    tu_mae += float((myy * (p - y).abs()).sum())
                    mau += float(myy.sum())
                val_loss = tu / (mau + EPS)
            lich_su.append(val_loss)
            # Dừng sớm theo đúng hàm mất mát đang huấn luyện, giống mọi lần chạy của tuần 3.
            if val_loss < tot_nhat - 1e-6:
                tot_nhat, xau = val_loss, 0
                val_mse_tot, val_mae_tot = tu_mse / (mau + EPS), tu_mae / (mau + EPS)
                trang_thai = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                xau += 1
                if xau >= PATIENCE:
                    break
        if trang_thai is not None:
            model.load_state_dict(trang_thai)
    thoi_gian = time.time() - t0

    model.eval()
    du_bao = []
    with torch.no_grad():
        for x, mxx, y, myy, s in loaders["test"]:
            du_bao.append(du_doan(model, x.to(DEVICE), mxx.to(DEVICE), s.to(DEVICE)).cpu().numpy())
    yhat = np.concatenate(du_bao)
    y_te = lat["test"].tensors[2].numpy()
    my_te = lat["test"].tensors[3].numpy()
    se, ae, dem = tong_theo_ho_so(yhat, y_te, my_te)
    ket = khoang_tin_cay(se, ae, dem, seed=seed)

    ten = nhan or (kind + "_neo" + anchor + "_" + loai_mat_mat + "_s" + str(seed))
    he_so_tinh = None
    if isinstance(model, LinearFamilyTinh):
        he_so_tinh = {
            "ten": TEN_TINH,
            "alpha_w": model.static_alpha.weight.detach().cpu().view(-1).tolist(),
            "alpha_b": float(model.static_alpha.bias.detach().cpu()),
            "beta_w": model.static_beta.weight.detach().cpu().view(-1).tolist(),
            "beta_b": float(model.static_beta.bias.detach().cpu()),
        }
    payload = {"name": ten, "kind": kind, "anchor": anchor, "seed": seed,
               "loai_mat_mat": loai_mat_mat, "huber_delta": huber_delta,
               "dung_tinh": bool(dung_tinh), "xao_tron_tinh": bool(xao_tron_tinh),
               "d_static": int(D_STATIC) if dung_tinh else 0,
               "seq_len": SEQ_LEN, "pred_len": PRED_LEN, "n_params": so_tham_so,
               "protocol": meta["protocol"], "read_tag": READ_TAG, "run_tag": RUN_TAG,
               "val_loss": tot_nhat if tot_nhat < float("inf") else None,
               "val_mse_obs": val_mse_tot, "val_mae_obs": val_mae_tot,
               "train_time_sec": round(thoi_gian, 2), "device": DEVICE,
               "history": lich_su, "test": ket, "he_so_tinh": he_so_tinh, "env": ENV}
    if ghi_file:
        thu_muc.mkdir(parents=True, exist_ok=True)
        (thu_muc / (ten + ".json")).write_text(json.dumps(payload, ensure_ascii=False, indent=2))
        np.savez_compressed(thu_muc / (ten + "_patient_sums.npz"), se=se, ae=ae, n=dem)
        if kind != "persistence":
            torch.save(model.state_dict(), thu_muc / (ten + ".pt"))
    print(ten.ljust(40), "MSE_obs", round(ket["mse"], 4), "| MAE_obs", round(ket["mae"], 4),
          "|", int(thoi_gian), "giây |", so_tham_so, "tham số")
    return payload


def doc_hoac_chay(ten, thu_muc=None, **tham_so):
    """Đọc lại lần huấn luyện đã có trong thư mục ghi, chưa có thì huấn luyện."""
    thu_muc = RUNS if thu_muc is None else thu_muc
    tep_json = thu_muc / (ten + ".json")
    tep_npz = thu_muc / (ten + "_patient_sums.npz")
    if tep_json.exists() and tep_npz.exists() and not CHAY_LAI_TU_DAU:
        d = json.loads(tep_json.read_text())
        print(ten.ljust(40), "đã có, đọc lại | MSE_obs", round(d["test"]["mse"], 4),
              "| MAE_obs", round(d["test"]["mae"], 4))
        return d
    return chay_mot_cau_hinh(nhan=ten, thu_muc=thu_muc, **tham_so)


print("Đã định nghĩa hàm huấn luyện")

Đã định nghĩa hàm huấn luyện


In [7]:
def hieu_so_bat_cap(se_a, ae_a, se_b, ae_b, dem, n_boot=2000, seed=0, alpha=0.05):
    """Khoảng tin cậy cho hiệu số hai mô hình, lấy mẫu lại cùng một danh sách bệnh nhân."""
    rng = np.random.default_rng(seed)
    N = len(dem)
    d_mse, d_mae = np.empty(n_boot), np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, N, N)
        mau = dem[idx].sum() + EPS
        d_mse[b] = (se_a[idx].sum() - se_b[idx].sum()) / mau
        d_mae[b] = (ae_a[idx].sum() - ae_b[idx].sum()) / mau
    lo, hi = 100 * alpha / 2, 100 * (1 - alpha / 2)
    mau_that = dem.sum() + EPS
    return {
        "hieu_mse": float((se_a.sum() - se_b.sum()) / mau_that),
        "hieu_mse_ci": [float(np.percentile(d_mse, lo)), float(np.percentile(d_mse, hi))],
        "hieu_mae": float((ae_a.sum() - ae_b.sum()) / mau_that),
        "hieu_mae_ci": [float(np.percentile(d_mae, lo)), float(np.percentile(d_mae, hi))],
        "p_hai_phia_mse": float(2 * min((d_mse <= 0).mean(), (d_mse >= 0).mean())),
        "p_hai_phia_mae": float(2 * min((d_mae <= 0).mean(), (d_mae >= 0).mean())),
    }


def nap_tong_tu_danh_sach(danh_sach_tep):
    # Lấy trung bình tổng sai số theo hồ sơ qua các hạt giống trước khi bắt cặp,
    # Vì biến thiên do hạt giống không phải đối tượng cần đo.
    ses, aes, dem = [], [], None
    for t in danh_sach_tep:
        t = Path(t)
        assert t.exists(), "Không thấy " + str(t)
        with np.load(t) as d:
            n = np.asarray(d["n"], dtype=np.float64)
            if dem is None:
                dem = n
            assert np.array_equal(dem, n), "Số điểm theo hồ sơ khác nhau giữa các hạt giống ở " + str(t)
            ses.append(np.asarray(d["se"], dtype=np.float64))
            aes.append(np.asarray(d["ae"], dtype=np.float64))
    return np.mean(ses, axis=0), np.mean(aes, axis=0), dem


def tong_diem_npz(duong):
    with np.load(duong) as z:
        return int(round(float(np.asarray(z["n"], dtype=np.float64).sum())))


def mse_mae_tu_npz(duong):
    with np.load(duong) as z:
        se, ae, n = (np.asarray(z[k], dtype=np.float64) for k in ("se", "ae", "n"))
    return float(se.sum() / (n.sum() + EPS)), float(ae.sum() / (n.sum() + EPS))


def thong_ke(v):
    a = np.asarray(v, dtype=np.float64)
    return float(a.mean()), (float(a.std(ddof=1)) if a.size > 1 else 0.0)


def phan_loai(ci):
    if float(ci[1]) < 0.0:
        return "bên trái thắng"
    if float(ci[0]) > 0.0:
        return "bên phải thắng"
    return "chưa phân biệt được"


def kiem_dinh_tu_mang(ma, cau_hoi, ten_trai, se_t, ae_t, dem_t, ten_phai, se_p, ae_p, dem_p,
                      so_hat_trai=0, so_hat_phai=0):
    """Kiểm định bắt cặp theo hồ sơ, bên trái trừ bên phải; âm nghĩa là bên trái tốt hơn."""
    assert np.array_equal(dem_t, dem_p), ma + ": hai bên không cùng số điểm theo hồ sơ"
    kd = hieu_so_bat_cap(se_t, ae_t, se_p, ae_p, dem_t, n_boot=N_BOOT,
                         seed=HAT_GIONG_LAY_MAU, alpha=MUC_ALPHA)
    r = {"ma": ma, "cau_hoi": cau_hoi, "ben_trai": ten_trai, "ben_phai": ten_phai,
         "so_hat_giong_trai": so_hat_trai, "so_hat_giong_phai": so_hat_phai,
         "so_ho_so": int(dem_t.size), "so_diem": int(round(float(dem_t.sum())))}
    r.update(kd)
    r["ket_luan_mse"] = phan_loai(r["hieu_mse_ci"])
    r["ket_luan_mae"] = phan_loai(r["hieu_mae_ci"])
    print(ma, "|", ten_trai, "trừ", ten_phai)
    print("   bình phương", round(r["hieu_mse"], 6), "| khoảng",
          [round(x, 6) for x in r["hieu_mse_ci"]], "| p", round(r["p_hai_phia_mse"], 4), "|", r["ket_luan_mse"])
    print("   tuyệt đối  ", round(r["hieu_mae"], 6), "| khoảng",
          [round(x, 6) for x in r["hieu_mae_ci"]], "| p", round(r["p_hai_phia_mae"], 4), "|", r["ket_luan_mae"])
    return r


def kiem_dinh_cap(ma, cau_hoi, ten_trai, tep_trai, ten_phai, tep_phai):
    se_t, ae_t, dem_t = nap_tong_tu_danh_sach(tep_trai)
    se_p, ae_p, dem_p = nap_tong_tu_danh_sach(tep_phai)
    return kiem_dinh_tu_mang(ma, cau_hoi, ten_trai, se_t, ae_t, dem_t, ten_phai, se_p, ae_p, dem_p,
                             len(tep_trai), len(tep_phai))


print("Đã định nghĩa hàm kiểm định bắt cặp")


def tach_theo_thuoc_do(r, ma_mse, ma_mae):
    # Tách một bản ghi bắt cặp thành hai phép kiểm định theo đúng mã đã khai báo trước.
    out = []
    for ma, ct in ((ma_mse, "mse"), (ma_mae, "mae")):
        out.append({"ma": ma, "thuoc_do": ct, "cau_hoi": r["cau_hoi"],
                    "ben_trai": r["ben_trai"], "ben_phai": r["ben_phai"],
                    "hieu": r["hieu_" + ct], "ci": r["hieu_" + ct + "_ci"],
                    "p_goc": r["p_hai_phia_" + ct], "ket_luan": r["ket_luan_" + ct],
                    "so_diem": r["so_diem"], "so_ho_so": r["so_ho_so"],
                    "so_hat_giong_trai": r["so_hat_giong_trai"],
                    "so_hat_giong_phai": r["so_hat_giong_phai"]})
    return out

Đã định nghĩa hàm kiểm định bắt cặp


In [8]:
def moc_theo_tag():
    """Mốc Persistence của từng lần chạy, dùng để loại lần chạy khác cách chia hoặc khác thang đo."""
    moc = {}
    for tep in sorted((OUT / "processed").glob("*/runs/persistence*.json")):
        try:
            d = json.loads(tep.read_text())
            moc.setdefault(str(d.get("run_tag", tep.parent.parent.name)), float(d["test"]["mse"]))
        except Exception:
            continue
    return moc


def tim_lan_da_co(kind, neo, mat_mat, hat_giong, tong_diem, huber_delta=1.0,
                  bo_qua_thu_muc=(), gom_tuan_4=False):
    """Tìm tệp tổng sai số theo hồ sơ của một cấu hình nền đã huấn luyện trên giao thức chính.

    Điều kiện ghép giống notebook 22 và notebook 19: cùng lần chạy nguồn, cùng độ dài hai cửa sổ, cùng mốc
    Persistence, cùng tổng số điểm chấm. Tuần 4 thêm ba điều kiện: bỏ mọi lần có nhánh tĩnh,
    với hàm Huber thì ngưỡng delta phải trùng, và mặc định bỏ các thư mục tuần 4 để đối chứng
    luôn là đúng các lần chạy đã báo cáo trong tuần 3.
    """
    moc = moc_theo_tag()
    ung_vien = []
    for tep in sorted((OUT / "processed").glob("*/runs/*.json")):
        thu_muc = tep.parent.parent.name
        if thu_muc in bo_qua_thu_muc:
            continue
        if thu_muc.startswith("w4_") and not gom_tuan_4:
            continue
        if "neolai" in tep.stem or "neotrung" in tep.stem or "sukien" in tep.stem:
            continue
        try:
            d = json.loads(tep.read_text())
        except Exception:
            continue
        if "test" not in d or d.get("dung_tinh", False):
            continue
        tag = str(d.get("run_tag", thu_muc))
        if "chia_seed" in tag:
            continue
        if tag in moc and abs(moc[tag] - MSE_MOC_KY_VONG) > 1e-9:
            continue
        if (d.get("kind") != kind or str(d.get("anchor")) != neo
                or str(d.get("loai_mat_mat", "mse")) != mat_mat
                or d.get("seed") != hat_giong):
            continue
        if mat_mat == "huber" and abs(float(d.get("huber_delta", 1.0)) - float(huber_delta)) > 1e-12:
            continue
        if (d.get("read_tag") != READ_TAG or d.get("seq_len") != SEQ_LEN
                or d.get("pred_len") != PRED_LEN):
            continue
        npz = tep.with_name(tep.stem + "_patient_sums.npz")
        if not npz.exists() or tong_diem_npz(npz) != tong_diem:
            continue
        ung_vien.append((tag, str(npz)))
    if not ung_vien:
        return None, None
    tag, duong = sorted(ung_vien)[-1]
    return Path(duong), tag


def lay_hoac_bu(kind, neo, mat_mat, hat_giong, huber_delta=1.0):
    """Lấy lần nền đã có trên giao thức chính; thiếu thì huấn luyện bù vào thư mục ghi."""
    tep, tag = tim_lan_da_co(kind, neo, mat_mat, hat_giong, TONG_DIEM_KY_VONG,
                             huber_delta=huber_delta, bo_qua_thu_muc=(RUN_TAG,))
    ten = kind + "_neo" + neo + "_" + mat_mat + "_s" + str(hat_giong)
    # Lần bù do W4_36 huấn luyện được W4_37 dùng lại, để hai sổ tay cùng một nền.
    tep_bu = OUT / "processed" / "w4_36_kiem_dinh_b_huber_va_giam_k5" / "runs" / (ten + "_patient_sums.npz")
    if tep is None and tep_bu.exists() and not CHAY_LAI_TU_DAU:
        tep, tag = tep_bu, "w4_36_kiem_dinh_b_huber_va_giam_k5"
        print(ten, "dùng lần bù của notebook 29")
    if tep is None:
        print("Chưa thấy", ten, "trên giao thức chính, huấn luyện bù")
        doc_hoac_chay(ten, kind=kind, anchor=neo, seed=hat_giong,
                      loai_mat_mat=mat_mat, huber_delta=huber_delta)
        tep, tag = RUNS / (ten + "_patient_sums.npz"), RUN_TAG
    assert tong_diem_npz(tep) == TONG_DIEM_KY_VONG, "Sai số điểm ở " + str(tep)
    return tep, tag


print("Đã định nghĩa hàm tìm lần chạy nền")

Đã định nghĩa hàm tìm lần chạy nền


## 4. Cổng giao thức trước khi huấn luyện

Tập kiểm tra phải có đúng 136.752 điểm có phép đo thật và mốc Persistence tính lại phải trùng mốc của tuần 2 tới chữ số thứ chín. Sai một trong hai điều kiện, sổ tay dừng ngay.

In [9]:
TONG_DIEM_TEST = int(round(float(SPLITS["test"].tensors[3].sum())))
print("Tổng số điểm chấm trên tập kiểm tra:", TONG_DIEM_TEST)
assert TONG_DIEM_TEST == TONG_DIEM_KY_VONG, (
    "Tập kiểm tra có " + str(TONG_DIEM_TEST) + " điểm, khác giao thức chính " +
    str(TONG_DIEM_KY_VONG) + " điểm")

p_per = doc_hoac_chay("persistence_luoi", kind="persistence", seed=0)
MSE_MOC = float(p_per["test"]["mse"])
MAE_MOC = float(p_per["test"]["mae"])
LECH_MOC = abs(MSE_MOC - MSE_MOC_KY_VONG)
print("Mốc Persistence:", MSE_MOC, "| lệch so với giá trị kỳ vọng:", LECH_MOC)
assert LECH_MOC < 1e-9, "Mốc Persistence khác giá trị kỳ vọng, kiểm tra lại dữ liệu đọc vào"
assert tong_diem_npz(RUNS / "persistence_luoi_patient_sums.npz") == TONG_DIEM_KY_VONG
print("Giao thức đánh giá khớp giao thức chính")

Tổng số điểm chấm trên tập kiểm tra: 136752
persistence_luoi                         MSE_obs 0.5788 | MAE_obs 0.4659 | 0 giây | 0 tham số
Mốc Persistence: 0.5787542076552471 | lệch so với giá trị kỳ vọng: 0.0
Giao thức đánh giá khớp giao thức chính


Ô dưới so thời điểm ghi tệp của gói hợp nhất (notebook 22 đến 26) với ba notebook bổ sung 19 đến 21. Gói hợp nhất phải mới hơn; nếu không, chạy lại notebook 22 đến 26 trước khi đọc kết quả cuối. Ô này cũng cảnh báo khi một trong ba notebook bổ sung chưa có kết quả trên Drive, hoặc khi gói hợp nhất chưa đọc đủ kết quả của cả ba.

In [10]:
W3_HOP = OUT / "processed" / "w3_artefacts_tong_hop" / "W3_HOP_NHAT_CHI_SO.json"
# Ba notebook bổ sung của tuần 3: mã nội bộ, mẫu thư mục kết quả và số thứ tự trong chuỗi tái lập.
NOTEBOOK_BO_SUNG = [("W3_31", "w3_31_*", "notebook 19"),
                    ("W3_33", "w3_33_*", "notebook 20"),
                    ("W3_32", "w3_32_*", "notebook 21")]
BO_SUNG, CHUA_CO_KET_QUA = [], []
for ma, mau, ten_nb in NOTEBOOK_BO_SUNG:
    tep = sorted((OUT / "processed").glob(mau + "/*.json"))
    if not tep:
        CHUA_CO_KET_QUA.append(ten_nb)
    BO_SUNG += tep

# Gói hợp nhất tự ghi danh sách notebook bổ sung đã đọc; thiếu mã nào nghĩa là kết quả đó chưa vào bảng xếp hạng.
CHUA_VAO_HOP_NHAT = []
if W3_HOP.exists():
    da_doc = set(json.loads(W3_HOP.read_text(encoding="utf-8")).get("so_tay_bo_sung_da_doc", []))
    CHUA_VAO_HOP_NHAT = [ten_nb for ma, mau, ten_nb in NOTEBOOK_BO_SUNG if ma not in da_doc]

THU_TU = {"co_goi_hop_nhat": W3_HOP.exists(), "tep_bo_sung": [str(p) for p in BO_SUNG],
          "notebook_bo_sung_chua_co_ket_qua": CHUA_CO_KET_QUA,
          "notebook_bo_sung_chua_vao_hop_nhat": CHUA_VAO_HOP_NHAT}
if CHUA_CO_KET_QUA:
    print("Cảnh báo: chưa thấy kết quả của", ", ".join(CHUA_CO_KET_QUA) +
          ". Hãy chạy các notebook này rồi chạy lại notebook 22 đến 26")
if CHUA_VAO_HOP_NHAT:
    print("Cảnh báo: gói hợp nhất chưa đọc kết quả của", ", ".join(CHUA_VAO_HOP_NHAT) +
          ". Hãy chạy lại notebook 22 đến 26")

if W3_HOP.exists() and BO_SUNG:
    t_hop = W3_HOP.stat().st_mtime
    t_bs = max(p.stat().st_mtime for p in BO_SUNG)
    THU_TU["goi_hop_nhat_luc"] = datetime.datetime.fromtimestamp(t_hop).isoformat(timespec="seconds")
    THU_TU["bo_sung_moi_nhat_luc"] = datetime.datetime.fromtimestamp(t_bs).isoformat(timespec="seconds")
    THU_TU["hop_nhat_moi_hon_bo_sung"] = bool(t_hop > t_bs) and not CHUA_CO_KET_QUA and not CHUA_VAO_HOP_NHAT
    print("Gói hợp nhất tuần 3 ghi lúc  :", THU_TU["goi_hop_nhat_luc"])
    print("Sổ tay bổ sung mới nhất lúc:", THU_TU["bo_sung_moi_nhat_luc"])
    if THU_TU["hop_nhat_moi_hon_bo_sung"]:
        print("Thứ tự chạy đúng: gói hợp nhất đã bao gồm kết quả của notebook 19, 20 và 21")
    elif t_hop <= t_bs:
        print("Cảnh báo: gói hợp nhất cũ hơn notebook 19, 20 và 21, cần chạy lại notebook 22 đến 26")
    else:
        print("Chưa xác nhận được thứ tự chạy vì gói hợp nhất còn thiếu kết quả bổ sung, xem cảnh báo ở trên")
else:
    THU_TU["hop_nhat_moi_hon_bo_sung"] = None
    print("Cảnh báo: thiếu gói hợp nhất hoặc thiếu tệp tổng hợp của notebook 19, 20 và 21")

Gói hợp nhất tuần 3 ghi lúc  : 2026-10-14T18:56:33
Sổ tay bổ sung mới nhất lúc: 2026-10-14T18:49:46
Thứ tự chạy đúng: gói hợp nhất đã bao gồm kết quả của notebook 19, 20 và 21


## 6. Mảng đặc trưng tĩnh

notebook 03 đã ghi mảng S cho từng hồ sơ: tuổi, giới tính, chiều cao và bốn cột mã hóa loại ICU, điền khuyết bằng trung bình rồi chuẩn hóa. Bốn đặc trưng lâm sàng trở thành bảy chiều, nên nhánh tĩnh của H2 có 2(7 + 1) = 16 tham số thêm.

In [11]:
TINH = {}
for ten in ("train", "val", "test"):
    S_ = SPLITS[ten].tensors[4].numpy()
    assert np.isfinite(S_).all(), "Mảng S của " + ten + " có giá trị không hữu hạn"
    TINH[ten] = {"so_ho_so": int(S_.shape[0]), "trung_binh": S_.mean(axis=0).round(4).tolist(),
                 "do_lech_chuan": S_.std(axis=0).round(4).tolist()}
S_tr = SPLITS["train"].tensors[4].numpy()
for j, t in enumerate(TEN_TINH):
    print(t.ljust(10), "trung bình", round(float(S_tr[:, j].mean()), 4), "| độ lệch chuẩn",
          round(float(S_tr[:, j].std()), 4), "| số mức giá trị", int(np.unique(S_tr[:, j]).size))
SO_THAM_SO_TINH = 2 * (D_STATIC + 1)
print("Số tham số thêm của nhánh tĩnh:", SO_THAM_SO_TINH)

Age        trung bình 0.0025 | độ lệch chuẩn 0.9981 | số mức giá trị 76
Gender     trung bình -0.0006 | độ lệch chuẩn 0.9997 | số mức giá trị 3
Height     trung bình 0.0004 | độ lệch chuẩn 0.7238 | số mức giá trị 68
ICUType_1  trung bình -0.0014 | độ lệch chuẩn 0.9986 | số mức giá trị 2
ICUType_2  trung bình 0.0015 | độ lệch chuẩn 1.0011 | số mức giá trị 2
ICUType_3  trung bình -0.0023 | độ lệch chuẩn 0.9994 | số mức giá trị 2
ICUType_4  trung bình 0.0022 | độ lệch chuẩn 1.0011 | số mức giá trị 2
Số tham số thêm của nhánh tĩnh: 16


## 7. Các lần chạy nền của tuần 3

NLinear neo B với hàm Huber ngưỡng 1,0 là cấu hình đề xuất của tuần 3 và là nền cho các phép kiểm định tuần 4. Hàm tìm kiếm bỏ qua mọi thư mục tuần 4, nên đối chứng luôn là đúng các lần chạy đã báo cáo.

In [12]:
NEN = {}
for mm in ("huber", "mse"):
    for s in range(N_SEEDS):
        tep, tag = tim_lan_da_co("nlinear", "B", mm, s, TONG_DIEM_KY_VONG, huber_delta=1.0)
        NEN["nlinear|B|" + mm + "|s" + str(s)] = {"tep": str(tep) if tep else None, "nguon": tag}
        print(("nlinear|B|" + mm + "|s" + str(s)).ljust(22), "| nguồn:", tag)
THIEU = [k for k, v in NEN.items() if v["tep"] is None]
print("Số lần nền còn thiếu:", len(THIEU), "(nếu thiếu, notebook 29 và notebook 30 sẽ huấn luyện bù)")

nlinear|B|huber|s0     | nguồn: bo_sung_14_huber_neo_b
nlinear|B|huber|s1     | nguồn: bo_sung_14_huber_neo_b
nlinear|B|huber|s2     | nguồn: bo_sung_14_huber_neo_b
nlinear|B|huber|s3     | nguồn: w3_17_luoi_nam_hat_giong
nlinear|B|huber|s4     | nguồn: w3_17_luoi_nam_hat_giong
nlinear|B|mse|s0       | nguồn: w3_18_chi_so_theo_nhom_kenh
nlinear|B|mse|s1       | nguồn: w3_18_chi_so_theo_nhom_kenh
nlinear|B|mse|s2       | nguồn: w3_18_chi_so_theo_nhom_kenh
nlinear|B|mse|s3       | nguồn: bo_sung_13_neo_abcd_nam_hat_giong
nlinear|B|mse|s4       | nguồn: bo_sung_13_neo_abcd_nam_hat_giong
Số lần nền còn thiếu: 0 (nếu thiếu, notebook 29 và notebook 30 sẽ huấn luyện bù)


## 8. Khai báo trước họ kiểm định tuần 4

Sáu phép kiểm định, hiệu chỉnh Holm trên đúng sáu phép này:

- K01, K02: NLinear neo B hàm Huber trừ mốc trọng số giảm K bằng 5, trên sai số bình phương và sai số tuyệt đối (notebook 29).
- K03, K04: cùng mô hình có nhánh tĩnh trừ mô hình không có nhánh tĩnh (notebook 30, giả thuyết H2).
- K05, K06: nhánh tĩnh thật trừ nhánh tĩnh nhận đặc trưng đã xáo trộn giữa các hồ sơ (notebook 30, đối chứng năng lực).

Quét ngưỡng Huber ở notebook 28 là bước chọn mô hình trên tập kiểm định, không phải phép kiểm định giả thuyết, nên không vào họ Holm. Các phép kiểm định chính giữ ngưỡng 1,0 đã dùng trong tuần 3 để kết quả không phụ thuộc vào một lựa chọn sinh ra sau khi đã nhìn dữ liệu kiểm tra.

In [13]:
HO_KIEM_DINH = [
    {"ma": "K01", "thuoc_do": "mse", "ben_trai": "nlinear|B|huber", "ben_phai": "giam_K5", "so_tay": "W4_36",
     "cau_hoi": "NLinear neo B hàm Huber có thắng mốc trọng số giảm K bằng 5 trên sai số bình phương không"},
    {"ma": "K02", "thuoc_do": "mae", "ben_trai": "nlinear|B|huber", "ben_phai": "giam_K5", "so_tay": "W4_36",
     "cau_hoi": "NLinear neo B hàm Huber có thắng mốc trọng số giảm K bằng 5 trên sai số tuyệt đối không"},
    {"ma": "K03", "thuoc_do": "mse", "ben_trai": "nlinear|B|huber|tinh", "ben_phai": "nlinear|B|huber", "so_tay": "W4_37",
     "cau_hoi": "Nhánh tĩnh có giảm sai số bình phương so với cùng mô hình không có nhánh tĩnh không"},
    {"ma": "K04", "thuoc_do": "mae", "ben_trai": "nlinear|B|huber|tinh", "ben_phai": "nlinear|B|huber", "so_tay": "W4_37",
     "cau_hoi": "Nhánh tĩnh có giảm sai số tuyệt đối so với cùng mô hình không có nhánh tĩnh không"},
    {"ma": "K05", "thuoc_do": "mse", "ben_trai": "nlinear|B|huber|tinh", "ben_phai": "nlinear|B|huber|tinh_xao_tron", "so_tay": "W4_37",
     "cau_hoi": "Lợi ích trên sai số bình phương có đến từ thông tin tĩnh thật chứ không chỉ từ tham số thêm không"},
    {"ma": "K06", "thuoc_do": "mae", "ben_trai": "nlinear|B|huber|tinh", "ben_phai": "nlinear|B|huber|tinh_xao_tron", "so_tay": "W4_37",
     "cau_hoi": "Lợi ích trên sai số tuyệt đối có đến từ thông tin tĩnh thật chứ không chỉ từ tham số thêm không"},
]
QUY_TAC = {
    "phuong_phap_hieu_chinh": "Holm",
    "so_phep": len(HO_KIEM_DINH),
    "muc_alpha": MUC_ALPHA,
    "n_boot": N_BOOT,
    "san_p": 1.0 / (N_BOOT + 1),
    "don_vi_lay_mau": "ho_so_benh_nhan",
    "so_hat_giong": N_SEEDS,
    "delta_cho_kiem_dinh_chinh": 1.0,
    "ket_luan_H2": {
        "ung_ho": "K03 và K05 cùng âm và cùng có p hiệu chỉnh Holm dưới mức alpha",
        "ung_ho_mot_phan": "K03 âm và có ý nghĩa sau Holm nhưng K05 không có ý nghĩa: lợi ích chưa tách được khỏi tham số thêm",
        "khong_ung_ho": "các trường hợp còn lại",
    },
    "ket_luan_moc_manh": "K01 và K02 cùng âm và cùng có ý nghĩa sau Holm thì NLinear neo B hàm Huber thắng mốc mạnh trên cả hai thang",
    "chon_delta": ("Chọn ngưỡng có MSE_obs trung bình trên tập kiểm định thấp nhất qua năm hạt giống; "
                   "hòa thì xét MAE_obs kiểm định. So sánh ngưỡng đã chọn với ngưỡng 1,0 trên tập kiểm tra "
                   "chỉ mang tính mô tả và không vào họ Holm"),
    "luoi_delta": [0.5, 1.0, 2.0],
    "doi_chieu_GraFITi": "Đối chiếu mô tả định nghĩa thước đo và giao thức, không phải so sánh đối đầu",
}
NOI_DUNG = {"ho_kiem_dinh": HO_KIEM_DINH, "quy_tac": QUY_TAC}
MA_BAM = hashlib.sha256(json.dumps(NOI_DUNG, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()

KHAI_BAO_DIR.mkdir(parents=True, exist_ok=True)
if TEP_KHAI_BAO.exists():
    cu = json.loads(TEP_KHAI_BAO.read_text(encoding="utf-8"))
    if cu["ma_bam"] == MA_BAM:
        print("Tệp khai báo đã có và trùng nội dung, giữ nguyên thời điểm khai báo", cu["khai_bao_luc"])
    else:
        assert CHO_PHEP_GHI_DE_KHAI_BAO, (
            "Nội dung khai báo khác bản đã khóa. Không ghi đè để giữ tính khai báo trước; "
            "chỉ bật CHO_PHEP_GHI_DE_KHAI_BAO khi chưa chạy notebook 28 đến notebook 30")
        (LOGS / ("khai_bao_cu_" + stamp + ".json")).write_text(json.dumps(cu, ensure_ascii=False, indent=1))
        TEP_KHAI_BAO.unlink()
if not TEP_KHAI_BAO.exists():
    ban_ghi = {"khai_bao_luc": datetime.datetime.now().isoformat(timespec="seconds"),
               "ma_bam": MA_BAM, **NOI_DUNG}
    TEP_KHAI_BAO.write_text(json.dumps(ban_ghi, ensure_ascii=False, indent=1), encoding="utf-8")
    print("Đã khóa khai báo trước | mã băm", MA_BAM[:16])

Đã khóa khai báo trước | mã băm 0b3b94e949f3dfb6


## 9. Ghi tệp cổng kiểm tra

In [14]:
CONG = {"kiem_tra_luc": datetime.datetime.now().isoformat(timespec="seconds"),
        "run_tag": RUN_TAG, "read_tag": READ_TAG, "giao_thuc": meta.get("protocol"),
        "tong_diem": TONG_DIEM_TEST, "mse_persistence": MSE_MOC, "mae_persistence": MAE_MOC,
        "thu_tu_tuan_3": THU_TU, "dac_trung_tinh": {"ten": TEN_TINH, "so_chieu": D_STATIC,
        "so_tham_so_nhanh_tinh": SO_THAM_SO_TINH, "thong_ke": TINH},
        "lan_chay_nen": NEN, "so_lan_nen_thieu": len(THIEU), "env": ENV}
DICH = RUN_DIR / "W4_cong_kiem_tra.json"
DICH.write_text(json.dumps(CONG, ensure_ascii=False, indent=1), encoding="utf-8")
print("Đã ghi", DICH.name)
print("Bước kế tiếp: chạy notebook 28")

Đã ghi W4_cong_kiem_tra.json
Bước kế tiếp: chạy notebook 28
